# Quickstart — ERK2 / 4FV7 interaction fingerprints

Sanity check that the new `aidd` package reproduces the protein–ligand IFP visualisation from `_archive/Week_3_Monday_Docking_and_Scoring.ipynb` on the same input data.

**Runtime:** under a minute on a laptop. Works on Windows / macOS / Colab.

**What it does:**
1. Loads the ERK2 receptor (PDB 4FV7) and its co-crystal ligand (E94).
2. Computes a ProLIF interaction fingerprint.
3. Renders the protein–ligand complex in 3D with `py3Dmol`.
4. Draws the 2D interaction network (`LigNetwork`).
5. Overlays the 3D interaction lines coloured by type.

If the figures look right, the package lift was successful and we can move on to step 7 (folding) and beyond.

## 1. Setup

In [1]:
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    # On Colab: pull repo + install pip extras. On local conda env: skip; deps already installed.
    !pip install -q rdkit datamol "prolif>=2.0" posebusters meeko py3Dmol biopython scikit-learn xgboost lightgbm
    REPO_ROOT = Path("/content/aidd-pipeline")
    if not REPO_ROOT.exists():
        !git clone https://github.com/hvmarco/aidd-pipeline.git {REPO_ROOT}
    sys.path.insert(0, str(REPO_ROOT / "src"))
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.insert(0, str(REPO_ROOT / "src"))

print(f"Repo root: {REPO_ROOT}")
print(f"Running on: {'Colab' if IS_COLAB else 'local'}")

Repo root: c:\Users\MarcoHernandez\Projects\AIinDD2025
Running on: local


In [2]:
# Auto-reload edits made to src/aidd/ without restarting the kernel.
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")  # ProLIF/RDKit/MDAnalysis are chatty

import prolif as plf

from aidd.ifp import compute_ifp, load_plf_molecule, to_wide_features
from aidd.viz import (
    show_protein_ligand,
    show_binding_site,
    show_interactions_3d,
    show_interaction_network,
)

print(f"prolif {plf.__version__}")

c:\Users\MarcoHernandez\.conda\envs\aidd\Lib\site-packages\MDAnalysis\topology\tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)


prolif 2.1.0


## 2. Inputs

In [3]:
PROTEIN = REPO_ROOT / "data" / "structures" / "erk2_4fv7.pdb"
LIGAND  = REPO_ROOT / "data" / "ligands"    / "erk2_4fv7_ref.pdb"
LIGAND_RESNAME = "E94"  # PDB three-letter code for the 4FV7 co-crystal ligand

assert PROTEIN.exists(), f"missing {PROTEIN}"
assert LIGAND.exists(),  f"missing {LIGAND}"
print(f"Protein: {PROTEIN.relative_to(REPO_ROOT)}")
print(f"Ligand:  {LIGAND.relative_to(REPO_ROOT)}")

Protein: data\structures\erk2_4fv7.pdb
Ligand:  data\ligands\erk2_4fv7_ref.pdb


## 3. Compute the interaction fingerprint

In [4]:
ifp_df = compute_ifp(PROTEIN, LIGAND)
print(f"Shape: {ifp_df.shape}  —  one row per pose, columns = (ligand_residue, protein_residue, interaction_type)")
ifp_df

Shape: (1, 16)  —  one row per pose, columns = (ligand_residue, protein_residue, interaction_type)


ligand        E94401.A                                               \
protein        ILE29.A    GLY30.A    GLU31.A    ALA50.A     LYS52.A   
interaction VdWContact VdWContact VdWContact VdWContact Hydrophobic   
Frame                                                                 
0                 True       True       True       True        True   

ligand                                                               \
protein                              ILE82.A   ILE101.A    GLN103.A   
interaction HBAcceptor VdWContact VdWContact VdWContact Hydrophobic   
Frame                                                                 
0                 True       True       True       True        True   

ligand                                                                         
protein                  ASP104.A   MET106.A              LYS112.A   LEU154.A  
interaction VdWContact VdWContact HBAcceptor VdWContact VdWContact VdWContact  
Frame                                                                          
0                 True       True       True       True       True       True

In [5]:
# Flattened view, ready for sklearn/XGBoost downstream
wide = to_wide_features(ifp_df)
print(f"{wide.shape[1]} features after flattening")
wide

16 features after flattening


,ALA50.A_VdWContact,ASP104.A_VdWContact,GLN103.A_Hydrophobic,GLN103.A_VdWContact,GLU31.A_VdWContact,GLY30.A_VdWContact,ILE101.A_VdWContact,ILE29.A_VdWContact,ILE82.A_VdWContact,LEU154.A_VdWContact,LYS112.A_VdWContact,LYS52.A_HBAcceptor,LYS52.A_Hydrophobic,LYS52.A_VdWContact,MET106.A_HBAcceptor,MET106.A_VdWContact
Frame,,,,,,,,,,,,,,,,
0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


## 4. 3D viewer — protein + ligand

Should match figure from `_archive/Week_3_Monday_Docking_and_Scoring.ipynb` (gold cartoon, cyan ligand sticks).

In [6]:
view = show_protein_ligand(PROTEIN, LIGAND, ligand_resname=LIGAND_RESNAME)
view.zoomTo({"resn": LIGAND_RESNAME})
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 5. 3D viewer — with binding-site residues highlighted

In [7]:
view = show_protein_ligand(PROTEIN, LIGAND, ligand_resname=LIGAND_RESNAME)
show_binding_site(view, ligand_resname=LIGAND_RESNAME, radius=5.0)
view.zoomTo({"resn": LIGAND_RESNAME})
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 6. 2D interaction network (ProLIF `LigNetwork`)

Shows all detected interactions in a flat schematic with the ligand at centre and interacting residues around it.

In [8]:
# Re-build the fingerprint object (we need it, not just the DataFrame, for LigNetwork)
protein_plf = load_plf_molecule(PROTEIN)
ligand_plf  = load_plf_molecule(LIGAND)

fp = plf.Fingerprint()
fp.run_from_iterable([ligand_plf], protein_plf, progress=False)

show_interaction_network(fp, ligand_plf)

## 7. 3D viewer — with interaction overlay

Lines coloured by interaction type (see `aidd.viz.INTERACTION_COLORS`). This is the figure to compare side-by-side with the course's Week 3 Monday output.

If anything looks off here (missing residues, wrong colours), that's a clue ProLIF 2.x changed the `fp.ifp[...]` dict shape since I wrote `show_interactions_3d`. Fix in `src/aidd/viz.py`.

In [9]:
view = show_protein_ligand(PROTEIN, LIGAND, ligand_resname=LIGAND_RESNAME)
show_interactions_3d(view, fp, protein_plf, ligand_plf, pose_index=0)
view.zoomTo({"resn": LIGAND_RESNAME})
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.